# Photo to Gaussian Splatting (.ply) in Colab

This notebook combines ideas from two selected candidates:

1. YassGan/3DGaussianSplatting-INRIA-Method-Colab  
   https://github.com/YassGan/3DGaussianSplatting-INRIA-Method-Colab
2. abolhoseinisina/Colab_COLMAP_3DGS  
   https://github.com/abolhoseinisina/Colab_COLMAP_3DGS

Pipeline:
- Input: photos in Google Drive
- COLMAP reconstruction (optional section if you already have COLMAP output)
- 3D Gaussian Splatting training
- Output: `point_cloud.ply`


## 0) Configuration

Set runtime to GPU in Colab: `Runtime -> Change runtime type -> T4 GPU`.

If you already have an undistorted COLMAP dataset, set `RUN_COLMAP = False` and place it under `COLMAP_UNDISTORTED_DIR`.

In [ ]:
RUN_COLMAP = True
IMAGE_SOURCE_DIR = "/content/drive/MyDrive/image_source"
PROJECT_DIR = "/content/project"
COLMAP_UNDISTORTED_DIR = f"{PROJECT_DIR}/undistorted"

# COLMAP feature extraction option:
# 1 if all photos are from one camera, 0 for mixed cameras.
SINGLE_CAMERA = 1

# 3DGS training options
MAX_IMAGE_RESOLUTION = 1600
TRAIN_ITERS = 30000
SAVE_ITERS = "7000 30000"

print("RUN_COLMAP:", RUN_COLMAP)
print("IMAGE_SOURCE_DIR:", IMAGE_SOURCE_DIR)
print("COLMAP_UNDISTORTED_DIR:", COLMAP_UNDISTORTED_DIR)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0.5) Optional: build a clean photo set from video

Use this section if you want to reconstruct from a video.

What this block does:
1. Extract frames with `ffmpeg`
2. Filter blurry frames
3. Filter too-dark / too-bright frames
4. Remove near-duplicates using perceptual hash
5. Use filtered frames as `IMAGE_SOURCE_DIR` for COLMAP

### Presets

Set `VIDEO_PRESET` in the config cell:
- `"street"` for outdoor city/building capture
- `"indoor"` for room-scale low-light scenes
- `"object"` for single-object capture
- `"custom"` to tune thresholds manually

In [ ]:
# Video preprocessing config
USE_VIDEO_INPUT = False
VIDEO_PATH = "/content/drive/MyDrive/input_video.mp4"
VIDEO_WORKDIR = "/content/video_pipeline"
RAW_FRAMES_DIR = f"{VIDEO_WORKDIR}/raw_frames"
FILTERED_FRAMES_DIR = f"{VIDEO_WORKDIR}/filtered_frames"

# Choose preset by scene type:
# - "street"   : outdoor walkthroughs / buildings / city
# - "indoor"   : rooms and low-light interiors
# - "object"   : single object on table / turntable-like capture
# - "custom"   : use manual thresholds below
VIDEO_PRESET = "street"

PRESETS = {
    "street": {
        "FPS_EXTRACT": 2.5,
        "MAX_FRAMES": 450,
        "LAPLACIAN_VAR_MIN": 140.0,
        "BRIGHTNESS_MIN": 40.0,
        "BRIGHTNESS_MAX": 225.0,
        "PHASH_MAX_DISTANCE": 7,
    },
    "indoor": {
        "FPS_EXTRACT": 2.0,
        "MAX_FRAMES": 400,
        "LAPLACIAN_VAR_MIN": 90.0,
        "BRIGHTNESS_MIN": 25.0,
        "BRIGHTNESS_MAX": 215.0,
        "PHASH_MAX_DISTANCE": 6,
    },
    "object": {
        "FPS_EXTRACT": 3.0,
        "MAX_FRAMES": 500,
        "LAPLACIAN_VAR_MIN": 170.0,
        "BRIGHTNESS_MIN": 45.0,
        "BRIGHTNESS_MAX": 220.0,
        "PHASH_MAX_DISTANCE": 5,
    },
}

# Manual fallback values (used only when VIDEO_PRESET = "custom")
FPS_EXTRACT = 2.0
MAX_FRAMES = 500
LAPLACIAN_VAR_MIN = 120.0
BRIGHTNESS_MIN = 35.0
BRIGHTNESS_MAX = 220.0
PHASH_MAX_DISTANCE = 6

if VIDEO_PRESET != "custom":
    if VIDEO_PRESET not in PRESETS:
        raise ValueError(f"Unknown VIDEO_PRESET: {VIDEO_PRESET}")
    cfg = PRESETS[VIDEO_PRESET]
    FPS_EXTRACT = cfg["FPS_EXTRACT"]
    MAX_FRAMES = cfg["MAX_FRAMES"]
    LAPLACIAN_VAR_MIN = cfg["LAPLACIAN_VAR_MIN"]
    BRIGHTNESS_MIN = cfg["BRIGHTNESS_MIN"]
    BRIGHTNESS_MAX = cfg["BRIGHTNESS_MAX"]
    PHASH_MAX_DISTANCE = cfg["PHASH_MAX_DISTANCE"]

print("USE_VIDEO_INPUT:", USE_VIDEO_INPUT)
print("VIDEO_PATH:", VIDEO_PATH)
print("FILTERED_FRAMES_DIR:", FILTERED_FRAMES_DIR)
print("VIDEO_PRESET:", VIDEO_PRESET)
print("FPS_EXTRACT:", FPS_EXTRACT)
print("MAX_FRAMES:", MAX_FRAMES)
print("LAPLACIAN_VAR_MIN:", LAPLACIAN_VAR_MIN)
print("BRIGHTNESS_MIN/BRIGHTNESS_MAX:", BRIGHTNESS_MIN, BRIGHTNESS_MAX)
print("PHASH_MAX_DISTANCE:", PHASH_MAX_DISTANCE)

In [ ]:
if USE_VIDEO_INPUT:
    !apt-get update
    !apt-get install -y ffmpeg
    !pip install -q opencv-python pillow imagehash numpy

In [ ]:
if USE_VIDEO_INPUT:
    import os
    import shutil

    os.makedirs(VIDEO_WORKDIR, exist_ok=True)
    if os.path.exists(RAW_FRAMES_DIR):
        shutil.rmtree(RAW_FRAMES_DIR)
    if os.path.exists(FILTERED_FRAMES_DIR):
        shutil.rmtree(FILTERED_FRAMES_DIR)
    os.makedirs(RAW_FRAMES_DIR, exist_ok=True)
    os.makedirs(FILTERED_FRAMES_DIR, exist_ok=True)

    # Extract frames from video
    !ffmpeg -y -i "{VIDEO_PATH}" -vf "fps={FPS_EXTRACT}" "{RAW_FRAMES_DIR}/frame_%06d.jpg"
    print("Raw frames extracted to:", RAW_FRAMES_DIR)

In [ ]:
if USE_VIDEO_INPUT:
    import os
    import glob
    import cv2
    import numpy as np
    from PIL import Image
    import imagehash
    import shutil

    def quality_score(lap_var: float, mean_brightness: float) -> float:
        # favor sharp frames and mid-range brightness
        brightness_penalty = abs(mean_brightness - 128.0)
        return lap_var - 0.2 * brightness_penalty

    frame_paths = sorted(glob.glob(os.path.join(RAW_FRAMES_DIR, "*.jpg")))
    print("Total raw frames:", len(frame_paths))

    accepted = []
    accepted_hashes = []

    for fp in frame_paths:
        img = cv2.imread(fp)
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        mean_brightness = float(gray.mean())

        if lap_var < LAPLACIAN_VAR_MIN:
            continue
        if mean_brightness < BRIGHTNESS_MIN or mean_brightness > BRIGHTNESS_MAX:
            continue

        ph = imagehash.phash(Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))

        is_dup = False
        for ph_prev in accepted_hashes:
            if (ph - ph_prev) <= PHASH_MAX_DISTANCE:
                is_dup = True
                break
        if is_dup:
            continue

        accepted_hashes.append(ph)
        accepted.append((fp, lap_var, mean_brightness, quality_score(lap_var, mean_brightness)))

    # Keep best frames by quality if too many
    accepted = sorted(accepted, key=lambda x: x[3], reverse=True)
    accepted = accepted[:MAX_FRAMES]

    # Restore chronological order for COLMAP
    accepted = sorted(accepted, key=lambda x: x[0])

    for i, (src, lap_var, mean_brightness, score) in enumerate(accepted, start=1):
        dst = os.path.join(FILTERED_FRAMES_DIR, f"img_{i:06d}.jpg")
        shutil.copy2(src, dst)

    print("Accepted frames:", len(accepted))
    print("Filtered frames dir:", FILTERED_FRAMES_DIR)

    if len(accepted) < 25:
        print("Warning: very few frames accepted. Consider lowering thresholds.")

In [ ]:
if USE_VIDEO_INPUT:
    # Redirect COLMAP input to filtered frames produced from video
    IMAGE_SOURCE_DIR = FILTERED_FRAMES_DIR
    print("IMAGE_SOURCE_DIR switched to:", IMAGE_SOURCE_DIR)

## 1) (Optional) Install and run COLMAP from photos

This section follows the COLMAP-focused notebook from candidate #4.

Skip this section if `RUN_COLMAP = False`.

In [ ]:
if RUN_COLMAP:
    !apt-get update
    !apt-get install -y \
      build-essential cmake git \
      libboost-all-dev libeigen3-dev libsuitesparse-dev \
      qtbase5-dev libglew-dev libglfw3-dev \
      libx11-dev libopencv-dev libgoogle-glog-dev \
      libgflags-dev libatlas-base-dev libopencv-core-dev \
      libopenimageio-dev openimageio-tools libopenexr-dev \
      libcgal-dev libcgal-qt5-dev libmetis-dev

In [ ]:
if RUN_COLMAP:
    %cd /content
    !rm -rf abseil-cpp
    !git clone https://github.com/abseil/abseil-cpp.git
    %cd /content/abseil-cpp
    !git checkout 20230802.0
    !rm -rf build && mkdir build
    %cd /content/abseil-cpp/build
    !cmake .. -DCMAKE_BUILD_TYPE=Release -DCMAKE_INSTALL_PREFIX=/usr/local -DCMAKE_POSITION_INDEPENDENT_CODE=ON
    !make -j$(nproc)
    !make install


In [ ]:
if RUN_COLMAP:
    %cd /content
    !rm -rf ceres-solver
    !git clone https://github.com/ceres-solver/ceres-solver.git
    %cd /content/ceres-solver
    !git checkout 2.1.0
    !rm -rf build && mkdir build
    %cd /content/ceres-solver/build
    !cmake .. -DBUILD_TESTING=OFF -DBUILD_EXAMPLES=OFF -DCMAKE_BUILD_TYPE=Release -DCMAKE_INSTALL_PREFIX=/usr/local
    !make -j$(nproc)
    !make install


In [ ]:
if RUN_COLMAP:
    %cd /content
    !rm -rf colmap
    !git clone https://github.com/colmap/colmap.git
    %cd /content/colmap
    !rm -rf build && mkdir build
    %cd /content/colmap/build
    !cmake .. -DCMAKE_BUILD_TYPE=Release -DCMAKE_INSTALL_PREFIX=/usr/local -DCUDA_ENABLED=OFF -DCeres_DIR=/usr/local/lib/cmake/Ceres -DAbsl_DIR=/usr/local/lib/cmake/absl
    !make -j$(nproc)
    !make install
    !colmap --help | head -n 20


In [ ]:
if RUN_COLMAP:
    %cd /content
    !rm -rf {PROJECT_DIR}
    !mkdir -p {PROJECT_DIR}/images
    !cp -r "{IMAGE_SOURCE_DIR}/." "{PROJECT_DIR}/images"

    !colmap feature_extractor \
      --database_path {PROJECT_DIR}/database.db \
      --image_path {PROJECT_DIR}/images \
      --ImageReader.single_camera {SINGLE_CAMERA} \
      --FeatureExtraction.use_gpu 0

    !colmap exhaustive_matcher \
      --database_path {PROJECT_DIR}/database.db \
      --FeatureMatching.use_gpu 0

    !mkdir -p {PROJECT_DIR}/sparse
    !colmap mapper \
      --database_path {PROJECT_DIR}/database.db \
      --image_path {PROJECT_DIR}/images \
      --output_path {PROJECT_DIR}/sparse

    !colmap image_undistorter \
      --image_path {PROJECT_DIR}/images \
      --input_path {PROJECT_DIR}/sparse/0 \
      --output_path {COLMAP_UNDISTORTED_DIR} \
      --output_type COLMAP \
      --max_image_size {MAX_IMAGE_RESOLUTION}

    !mkdir -p "{COLMAP_UNDISTORTED_DIR}/sparse/0"
    !mv {COLMAP_UNDISTORTED_DIR}/sparse/*.bin {COLMAP_UNDISTORTED_DIR}/sparse/0/ || true

    print("COLMAP outputs ready at:", COLMAP_UNDISTORTED_DIR)


## 2) Install 3DGS environment (Colab compatibility)

This section follows candidate #2 style for Colab compatibility.

In [ ]:
!wget -O mini.sh https://repo.anaconda.com/miniconda/Miniconda3-py37_23.1.0-1-Linux-x86_64.sh
!chmod +x mini.sh
!bash ./mini.sh -b -f -p /usr/local
!conda install -q -y python=3.7
import sys
sys.path.append('/usr/local/lib/python3.7/site-packages')
!python --version


In [ ]:
!wget -q https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
!chmod +x cuda_11.8.0_520.61.05_linux.run
!./cuda_11.8.0_520.61.05_linux.run --silent --toolkit --no-drm --no-man-page
import os
os.environ['PATH'] += ':/usr/local/cuda-11.8/bin'
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.8/lib64:/usr/lib64-nvidia'
!nvcc --version


In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==1.12.1+cu116 torchvision==0.13.1+cu116 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu116

import torch
print("torch.cuda.is_available:", torch.cuda.is_available())
print("torch.version.cuda:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
!nvidia-smi


In [ ]:
%cd /content
!rm -rf gaussian-splatting
!git clone --recursive https://github.com/camenduru/gaussian-splatting
!pip install -q plyfile
%cd /content/gaussian-splatting
!pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q /content/gaussian-splatting/submodules/simple-knn


## 3) Train Gaussian Splatting

Expected dataset structure for `COLMAP_UNDISTORTED_DIR`:

```
undistorted/
  images/
  sparse/0/
    cameras.bin
    images.bin
    points3D.bin
```


In [ ]:
%cd /content/gaussian-splatting
!python train.py -s {COLMAP_UNDISTORTED_DIR} --iterations {TRAIN_ITERS} --save_iterations {SAVE_ITERS}


In [ ]:
# Find newest model dir and check generated .ply files
import os, glob

model_dirs = [d for d in glob.glob('/content/gaussian-splatting/output/*') if os.path.isdir(d)]
model_dirs = sorted(model_dirs, key=lambda p: os.path.getmtime(p))
assert model_dirs, "No model directory found in /content/gaussian-splatting/output"
MODEL_DIR = model_dirs[-1]

ply_files = glob.glob(os.path.join(MODEL_DIR, '**', '*.ply'), recursive=True)
print('MODEL_DIR:', MODEL_DIR)
print('PLY files:')
for p in ply_files:
    print(' -', p)

assert ply_files, 'No .ply file found. Check training logs.'
POINT_CLOUD_PLY = sorted(ply_files, key=lambda p: os.path.getmtime(p))[-1]
print('Selected POINT_CLOUD_PLY:', POINT_CLOUD_PLY)


In [ ]:
# Optional: render train/test views (replace with your actual model dir if needed)
%cd /content/gaussian-splatting
!python render.py -s {COLMAP_UNDISTORTED_DIR} -m {MODEL_DIR}


In [ ]:
# Copy output to Google Drive
EXPORT_DIR = '/content/drive/MyDrive/gaussian_splatting_output'
!mkdir -p {EXPORT_DIR}
!cp -r {MODEL_DIR} {EXPORT_DIR}/
print('Copied model folder to:', EXPORT_DIR)
print('Main ply:', POINT_CLOUD_PLY)


## Notes

- If training crashes because of memory, try fewer input images or lower image resolution in COLMAP undistortion.
- If your COLMAP scene has no `/sparse/0`, inspect mapper logs and input image quality/overlap.
- If you already have COLMAP-ready data, set `RUN_COLMAP = False` and point `COLMAP_UNDISTORTED_DIR` to that dataset.
- The expected final asset is `point_cloud.ply` inside the selected model directory.